### Noise_linear_regression

In [ ]:
from statsmodels.formula.api import ols

# model = ols(
#     "total_score ~ anxious + calm + conventional + critical + dependable + disorganized + enthusiastic + experiences + reserved + sympathetic + hours + experience",
#     data=merged_class,
# ).fit()

# model = ols(
#     "total_score ~ gpa_all + hours + disorganized + level", data=merged_social
# ).fit()

# model = ols(
#     "total_score ~ anxious + gpa_all + cs_65 + due + sleep_quality", data=merged_social
# ).fit()

# model = ols(
#     "total_score ~ gpa_all + anxious + number + happyornot", data=merged_social
# ).fit()

model = ols(
    "total_score ~ gpa_all + hours + sleep_hours + walk + cs_65 + anxious + exercise",
    data=merged_social,
).fit()

model.summary()

/opt/miniconda3/lib/python3.12/site-packages/scipy/stats/_axis_nan_policy.py:531: UserWarning: kurtosistest only valid for n>=20 ... continuing anyway, n=19
  res = hypotest_fun_out(*samples, **kwds)


<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:            total_score   R-squared:                       0.746
Model:                            OLS   Adj. R-squared:                  0.584
Method:                 Least Squares   F-statistic:                     4.616
Date:                Tue, 22 Apr 2025   Prob (F-statistic):             0.0123
Time:                        00:12:08   Log-Likelihood:                -47.801
No. Observations:                  19   AIC:                             111.6
Df Residuals:                      11   BIC:                             119.2
Df Model:                           7                                         
Covariance Type:            nonrobust                                         
===============================================================================
                  coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------
Intercept      35.0218     17.869      1.960      0.076      -4.307      74.351
gpa_all       -10.7751      2.882     -3.739      0.003     -17.119      -4.432
hours           1.6131      0.941      1.715      0.114      -0.457       3.683
sleep_hours    -0.8218      1.305     -0.630      0.542      -3.693       2.050
walk           -2.2430      1.931     -1.161      0.270      -6.494       2.008
cs_65           3.1896      1.750      1.822      0.096      -0.663       7.042
anxious         3.4872      2.251      1.549      0.150      -1.468       8.443
exercise       -2.3223      1.678     -1.384      0.194      -6.015       1.370
==============================================================================
Omnibus:                        7.146   Durbin-Watson:                   2.084
Prob(Omnibus):                  0.028   Jarque-Bera (JB):                4.511
Skew:                           1.090   Prob(JB):                        0.105
Kurtosis:                       3.972   Cond. No.                         201.
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

In [ ]:
from sklearn.metrics import mean_squared_error, r2_score

# Extract true and predicted values
y_true = merged_social['total_score']
y_pred = model.fittedvalues  # 注意是无噪声的model

# Calculate MSE
baseline_mse = mean_squared_error(y_true, y_pred)

# Calculate R²
baseline_r2 = r2_score(y_true, y_pred)

print("Baseline MSE:", baseline_mse)
print("Baseline R²:", baseline_r2)


Baseline MSE: 8.969175169954072
Baseline R²: 0.7460093947008613


In [ ]:
# add noise to sleep_hours, gpa_all, anxious
import numpy as np
from statsmodels.formula.api import ols

# Make a copy of the original dataset to avoid modifying the original data
noisy_merged_social = merged_social.copy()

# Add Gaussian noise to gpa_all (mean=0, std=0.05)
gpa_noise = np.random.normal(0, 0.05, size=noisy_merged_social.shape[0])
noisy_merged_social['gpa_all'] += gpa_noise

# Add Gaussian noise to sleep_hours (mean=0, std=1), then round and clip to [0, 24]
sleep_noise = np.random.normal(0, 1, size=noisy_merged_social.shape[0])
noisy_merged_social['sleep_hours'] = noisy_merged_social['sleep_hours'] + sleep_noise
noisy_merged_social['sleep_hours'] = noisy_merged_social['sleep_hours'].round().clip(0, 24)

# Add discrete noise (±1) to anxious, clip the values to [1, 5]
anxious_noise = np.random.choice([-1, 0, 1], size=noisy_merged_social.shape[0])
noisy_merged_social['anxious'] = (noisy_merged_social['anxious'] + anxious_noise).clip(1, 5)

# Fit the OLS model using the noisy dataset
high_relevent_noisy_model = ols(
    "total_score ~ gpa_all + hours + sleep_hours + walk + cs_65 + anxious + exercise",
    data=noisy_merged_social
).fit()

# Display the regression results
print(high_relevent_noisy_model.summary())



                            OLS Regression Results                            
Dep. Variable:            total_score   R-squared:                       0.634
Model:                            OLS   Adj. R-squared:                  0.401
Method:                 Least Squares   F-statistic:                     2.725
Date:                Tue, 22 Apr 2025   Prob (F-statistic):             0.0667
Time:                        00:26:25   Log-Likelihood:                -51.266
No. Observations:                  19   AIC:                             118.5
Df Residuals:                      11   BIC:                             126.1
Df Model:                           7                                         
Covariance Type:            nonrobust                                         
                  coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------
Intercept      28.8239     16.310      1.767      

/opt/miniconda3/lib/python3.12/site-packages/scipy/stats/_axis_nan_policy.py:531: UserWarning: kurtosistest only valid for n>=20 ... continuing anyway, n=19
  res = hypotest_fun_out(*samples, **kwds)


In [ ]:
high_relevant_y_true = merged_social['total_score']
high_relevant_y_pred = high_relevent_noisy_model.fittedvalues  

from sklearn.metrics import mean_squared_error,r2_score

mse = mean_squared_error(high_relevant_y_true, high_relevant_y_pred)
print("MSE for noised model with highly relevant variables:", mse)

r2 = r2_score(high_relevant_y_true, high_relevant_y_pred)
print("R² for noised model with highly relevant variables:", r2)


MSE for noised model with highly relevant variables: 12.917311305839903
R² for noised model with highly relevant variables: 0.634205414072152


In [ ]:

noisy_less_relevant_social = merged_social.copy()

# Add noise to hours
hours_noise = np.random.normal(0, 1, size=noisy_less_relevant_social.shape[0])
noisy_less_relevant_social['hours'] = noisy_less_relevant_social['hours'] + hours_noise
noisy_less_relevant_social['hours'] = noisy_less_relevant_social['hours'].round()

# Add noise to walk
walk_noise = np.random.normal(0, 1, size=noisy_less_relevant_social.shape[0])
noisy_less_relevant_social['walk'] = noisy_less_relevant_social['walk'] + walk_noise
noisy_less_relevant_social['walk'] = noisy_less_relevant_social['walk'].round()

# Add noise to cs_65
cs65_noise = np.random.normal(0, 1, size=noisy_less_relevant_social.shape[0])
noisy_less_relevant_social['cs_65'] = noisy_less_relevant_social['cs_65'] + cs65_noise
noisy_less_relevant_social['cs_65'] = noisy_less_relevant_social['cs_65'].round()

# Fit the OLS model again with noise
noisy_less_relevant_model = ols(
    "total_score ~ gpa_all + hours + sleep_hours + walk + cs_65 + anxious + exercise",
    data=noisy_less_relevant_social
).fit()

# Show the result
print(noisy_less_relevant_model.summary())


                            OLS Regression Results                            
Dep. Variable:            total_score   R-squared:                       0.722
Model:                            OLS   Adj. R-squared:                  0.545
Method:                 Least Squares   F-statistic:                     4.080
Date:                Tue, 22 Apr 2025   Prob (F-statistic):             0.0190
Time:                        00:35:55   Log-Likelihood:                -48.661
No. Observations:                  19   AIC:                             113.3
Df Residuals:                      11   BIC:                             120.9
Df Model:                           7                                         
Covariance Type:            nonrobust                                         
                  coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------
Intercept      26.4189     17.811      1.483      

/opt/miniconda3/lib/python3.12/site-packages/scipy/stats/_axis_nan_policy.py:531: UserWarning: kurtosistest only valid for n>=20 ... continuing anyway, n=19
  res = hypotest_fun_out(*samples, **kwds)


In [ ]:
less_relevant_y_true = merged_social['total_score']
less_relevant_y_pred = noisy_less_relevant_model.fittedvalues  

from sklearn.metrics import mean_squared_error,r2_score

mse = mean_squared_error(less_relevant_y_true, less_relevant_y_pred)
print("MSE for noised model with less relevant variables:", mse)

r2 = r2_score(less_relevant_y_true, less_relevant_y_pred)
print("R² for noised model with less relevant variables:", r2)

MSE for noised model with less relevant variables: 9.819126456867306
R² for noised model with less relevant variables: 0.7219403317438737


For Linear Regression model:

| Condition | MSE | RMSE (√MSE) | R² |
|:---|:---|:---|:---|
| No Noise | 8.9692 | 2.9949 | 0.7460 |
| Noise on Highly Relevant Variables | 12.9173 | 3.5935 | 0.6345 |
| Noise on Less Relevant Variables | 9.8191 | 3.1344 | 0.7219 |

